# Black-Litterman Portfolio Optimization

This notebook implements the **Black-Litterman (BL) model** as a stronger baseline for the RL-based portfolio optimization project.

The BL model addresses a key weakness of raw Markowitz optimization — extreme sensitivity to small estimation errors in expected returns — by blending:

1. **Market equilibrium returns** (implied by CAPM / market-cap weights), and
2. **Investor views** (subjective, absolute, or relative) expressed with a confidence level.

The result is a posterior distribution over expected returns that is more stable and intuitive than the sample mean.

### References
- Black, F. & Litterman, R. (1992). *Global Portfolio Optimization*. Financial Analysts Journal.
- He, G. & Litterman, R. (1999). *The Intuition Behind Black-Litterman Model Portfolios*. Goldman Sachs.
- [Portfolio Optimization Book (Roncalli), Ch. 4](https://portfoliooptimizationbook.com/portfolio-optimization-book.pdf)

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import scipy.optimize as sco

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Download Price Data & Market-Cap Weights

In [ ]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'GS', 'XOM', 'JNJ']
START_DATE = '2018-01-01'
END_DATE   = '2023-12-31'
TRADING_DAYS = 252
RF = 0.04   # risk-free rate (annualised)

prices = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)['Close']
prices.dropna(inplace=True)

returns = np.log(prices / prices.shift(1)).dropna()
Sigma = returns.cov().values * TRADING_DAYS   # annualised covariance matrix

# Approximate market-cap weights (proxy: end-of-period price * arbitrary share count)
# In practice, fetch actual market-cap data.  Here we use equal weights as a placeholder.
n = len(TICKERS)
w_mkt = np.ones(n) / n   # replace with actual market-cap weights

print(f'Assets: {TICKERS}')
print(f'Market weights: {w_mkt.round(3)}')

## 3. CAPM Implied (Equilibrium) Returns

The Black-Litterman prior is derived from the equilibrium expected returns implied by CAPM:

$$\Pi = \delta \Sigma w_{mkt}$$

where $\delta$ is the risk-aversion coefficient and $w_{mkt}$ are the market-cap weights.

In [ ]:
# Risk-aversion coefficient (commonly estimated as (E[Rm]-Rf)/Var(Rm))
market_return = w_mkt @ (returns.mean().values * TRADING_DAYS)
market_var = w_mkt @ Sigma @ w_mkt
delta = (market_return - RF) / market_var
print(f'Implied risk-aversion delta: {delta:.3f}')

# Equilibrium (prior) expected returns
Pi = delta * Sigma @ w_mkt

print('\nEquilibrium expected returns (annualised):')
for ticker, pi in zip(TICKERS, Pi):
    print(f'  {ticker}: {pi:.2%}')

## 4. Express Investor Views

Views are encoded as:
- **P** — a $(k \times n)$ matrix mapping views to assets  
- **Q** — a $(k \times 1)$ vector of expected return for each view  
- **Omega** — a $(k \times k)$ diagonal uncertainty matrix

**Example views used here:**
1. AAPL will outperform MSFT by 2% per year (relative view).
2. XOM will return 5% per year (absolute view).

In [ ]:
# View 1: AAPL outperforms MSFT by 2%
# View 2: XOM returns 5% absolutely
ticker_idx = {t: i for i, t in enumerate(TICKERS)}

P = np.zeros((2, n))
P[0, ticker_idx['AAPL']] =  1
P[0, ticker_idx['MSFT']] = -1
P[1, ticker_idx['XOM']]  =  1

Q = np.array([0.02, 0.05])   # 2% relative, 5% absolute

# Uncertainty: proportional to variance of implied returns
tau = 0.05   # scaling factor (small → high confidence in equilibrium)
Omega = np.diag(np.diag(tau * P @ Sigma @ P.T))

print('P matrix (views × assets):')
print(pd.DataFrame(P, columns=TICKERS).round(2))
print('\nQ (view returns):', Q)
print('Omega (view uncertainty):', np.diag(Omega).round(6))

## 5. Black-Litterman Posterior

The BL posterior combines the prior $\Pi$ with the views:

$$\mu_{BL} = \left[(\tau\Sigma)^{-1} + P^T \Omega^{-1} P\right]^{-1} \left[(\tau\Sigma)^{-1}\Pi + P^T \Omega^{-1} Q\right]$$

$$M^{-1} = (\tau\Sigma)^{-1} + P^T\Omega^{-1}P \quad \text{(posterior covariance)}$$

In [ ]:
tau_Sigma = tau * Sigma
tau_Sigma_inv = np.linalg.inv(tau_Sigma)
Omega_inv = np.linalg.inv(Omega)

# Posterior precision and mean
M_inv = tau_Sigma_inv + P.T @ Omega_inv @ P
M = np.linalg.inv(M_inv)

mu_bl = M @ (tau_Sigma_inv @ Pi + P.T @ Omega_inv @ Q)

print('Black-Litterman expected returns vs. equilibrium prior:')
comparison = pd.DataFrame({'Equilibrium (Pi)': Pi, 'BL Posterior': mu_bl}, index=TICKERS)
print(comparison.map(lambda x: f'{x:.2%}'))

## 6. Optimal Portfolio using BL Returns

In [ ]:
# Use Sigma + M as the posterior covariance (accounts for parameter uncertainty)
Sigma_bl = Sigma + M

def neg_sharpe_bl(weights):
    ret = weights @ mu_bl
    vol = np.sqrt(weights @ Sigma_bl @ weights)
    return -(ret - RF) / vol

constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
bounds = tuple((0, 1) for _ in range(n))
w0 = np.ones(n) / n

result = sco.minimize(neg_sharpe_bl, w0, method='SLSQP', bounds=bounds, constraints=constraints)
w_bl = result.x

print('Black-Litterman Maximum-Sharpe Portfolio:')
for ticker, w in zip(TICKERS, w_bl):
    print(f'  {ticker}: {w:.2%}')
ret_bl = w_bl @ mu_bl
vol_bl = np.sqrt(w_bl @ Sigma_bl @ w_bl)
print(f'  Expected Return: {ret_bl:.2%}')
print(f'  Volatility: {vol_bl:.2%}')
print(f'  Sharpe Ratio: {(ret_bl - RF) / vol_bl:.3f}')

## 7. Compare BL vs. Markowitz Allocations

In [ ]:
# Markowitz max-Sharpe for comparison
mu_sample = returns.mean().values * TRADING_DAYS

def neg_sharpe_mvo(weights):
    ret = weights @ mu_sample
    vol = np.sqrt(weights @ Sigma @ weights)
    return -(ret - RF) / vol

res_mvo = sco.minimize(neg_sharpe_mvo, w0, method='SLSQP', bounds=bounds, constraints=constraints)
w_mvo = res_mvo.x

comparison_w = pd.DataFrame({'Markowitz': w_mvo, 'Black-Litterman': w_bl}, index=TICKERS)
comparison_w.plot(kind='bar', title='Portfolio Weights: Markowitz vs Black-Litterman', ylabel='Weight')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Next Steps

- Incorporate **real market-cap weights** from a data provider.
- Experiment with different view specifications and confidence levels.
- Use the BL portfolio as a **starting policy** or **benchmark** for the RL agents.
- Evaluate BL performance on rolling out-of-sample windows and compare with RL agents.